# 02 Context and State

So far we have given our agents access to a lot of information & functionality via:
* **Prompts** -> system prompts via the `system_prompt` parameter of the `create_agent()` function and user-prompts when we `invoke()` the agent using the `{"messages" : {"role": "user", "context": query}]}` parameter to `invoke()`
* **Tools** -> custom tools we created and passed in via the `tools=[...]` parameter of the `create_agent()` call.
* **Chat Memory** -> via an instance of `langgraph.checkpoint.memory.InMemorySaver` class passed via the `checkpointer` parameter of `create_agent()` call
* **MCP Servers** -> Access to additional 3rd party tools via MCP Servers

What if you wanted to pass in additional information that the agent needs to know, which it can share with it's tools? For example, let's suppose you are developing a text-to-SQL agent, which converts your text query to a SQL. Such an Agent would need access to a database connection and a relevant schema to convert the text to SQL. You would need to pass in the database connection (& perhaps the schema too) as additional information to your agent, which it uses during the `invoke()` or `stream()` calls.

As another example, let's say you wanted your agent to know user preferences such as preferred language (English, Spanish, Hindi) to return responses in, or certain information about you - say you are from India (so all tools that return numbers & currency should format it per Indian locale for example).

Such additional information is usually passed in as an instance of a `dataclass` (or you could even a `TypedDict` class, which should be familiar to LangGraph developers) or a class derived from Pydantic's `BaseClass`. Here is an example of how you would use it.

```python
# when using dataclasses
from dataclasses import dataclass

# annotate it with @dataclass, use any class name of your choice
@dataclass
class UserInfo:
    # you can pass in any attributes, after all what info
    # you want to pass is upto you
    locale : str
    language : str

# ------------------------------
# OR if you prefer, you can also use a TypedDict
from typing import TypedDict

class UserInfo(TypedDict):
    # you can pass in any attributes, after all what info
    # you want to pass is upto you
    locale : str
    language : str

```
And this is how you pass it into your agent

```python
from langchain.agents import create_agent

# instantiats your context
context = UserInfo(locale="India", language="Hindi")

agent = create_agent(
    model="openai:gpt-5-nano',
    system_prompt="...",
    tools=[get_flight_info,...],
    ....
    # and here is the context
    context_schema=UserInfo,
)
```

Surprise, surprise! The `context_schema` is not directly useful to the Agent, but it is very useful to all the tools. In any tool function, you can use `langgraph.runtime.get_runtime()` function to get instance of the context you had passed into the agent. Here is an example of how you can access the context in a tool

```python
from langchain.tools import tool
from langgraph.runtime import get_context

@tool
def get_flight_info(from_city: str, to_city: str, date: str) -> str:
    """ returns flight info & price for source & destination cities for a data """
    flight_info = call_some_api_to_get_flight_info(from_city, to_city, date)

    context = get_context()
    # if context is an instance of dataclass, you can access attributes as follows
    locale = context.locale
    language = context.language
    # alternatively, if context is an instance of TypedDict, the access attribs as follows
    locale = context["locale"]
    language = context["language"]
    # now use them in any way you want in the function, for example
    # (assume that format_to_locale() funtion will format currency to locale specific format
    # for example "123456" will be formatted as "1,23,456" for India locale
    flight_info["price"] = format_to_locale(flight_info["price"], locale)  
    # format flight info as a string
    flight_info_str = format_as_str(flight_info)
    return flight_info_str
```  

There is another simpler way to pass around the context to your tool functions.
* You define your context as a `dataclass` annotated with a `@dataclass` decorator as above
* Add a `runtime: ToolRuntime` parameter to your tool functions as shown below
* Pass in an instance of your dataclass to the `agent.invoke()` method

Here is an example

```python
# define your tool functions like this

from langchain.tools import tool, ToolRuntime

@tool
def get_flight_info(from_city: str, to_city: str, date: str, runtime: ToolRuntime) -> str:
    """ returns flight info & price for source & destination cities for a data """
    flight_info = call_some_api_to_get_flight_info(from_city, to_city, date)
    # access context attributes like this
    locale = runtime.context.locale
    language = runtime.context.language   
    # and use them as before...

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="...",
    tools=[get_flight_details],
    ...
)

context = UserInfo(locale="India", language="Hindi")

response = agent.invoke(
    {"messages" : {"role":"user", "content":"<<your query>>"}},
    # pass in the context here
    context=context,
)
```

In this workbook, we'll show a very simple example of runtime context that lists my favourite & least-favourite color.

In [28]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

In [29]:
from dataclasses import dataclass


@dataclass
class ColorContext:
    favourite_color: str = "Blue"
    least_favourite_color: str = "yellow"

In [30]:
from langchain.tools import tool, ToolRuntime


@tool
def favourite_color(runtime: ToolRuntime) -> str:
    """gets favourite color of user"""
    print(f" -------- favourite_color({runtime.context}) tool called --------")
    return runtime.context.favourite_color


@tool
def least_favourite_color(runtime: ToolRuntime) -> str:
    """gets favourite color of user"""
    print(f" -------- least_favourite_color({runtime.context}) tool called --------")
    return runtime.context.least_favourite_color

In [31]:
# create our agent
from langchain.agents import create_agent

color_pref = ColorContext(favourite_color="blue", least_favourite_color="yellow")

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=(
        """
        You are a helpful agent that can answer information about my color preference.
        You have access to the following tools:
            - favourite_color : to get my favourite color
            - least_favourite_color : to get my least favourite color
        Use these tools for any question regarding my color preference & return response as
            "Your [least]favourite color is <<color>>"
        For all other queries about me, respond as follows:
            "Sorry, this information is confidential. I can only tell you about your color preference"
        """
    ),
    tools=[favourite_color, least_favourite_color],
)

In [32]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "What is my favourite color?"}},
    context=color_pref,
)
console.print(f"[bright_blue]{response["messages"][-1].content}[/bright_blue]")

c:\Dev\Code\git-projects\learning_langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


 -------- favourite_color(ColorContext(favourite_color='blue', least_favourite_color='yellow')) tool called --------


Your favourite color is blue

In [33]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "What is my least favourite color?"}},
    context=color_pref,
)
console.print(f"[bright_blue]{response["messages"][-1].content}[/bright_blue]")

c:\Dev\Code\git-projects\learning_langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


 -------- least_favourite_color(ColorContext(favourite_color='blue', least_favourite_color='yellow')) tool called --------


Your least favourite color is yellow

In [34]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "What is my age?"}},
    context=color_pref,
)
console.print(f"[bright_blue]{response["messages"][-1].content}[/bright_blue]")

Sorry, this information is confidential. I can only tell you about your color preference

### The context we pass in above is IMMUTABLE (serves as read-only info to agent & tools)

But what if we want to pass around something that the agent can _update_? If you recall, this is possible using the `InMemorySaver()` class and passed in to the `create_agent()` via its `checkpointer=InMemoryState()` parameter. However, so far the agent was updating this automatically without our intervention.

We can also add our own _custom fields_ to this context as shown in the example below:

1. We have to define an instance of the `langchain.agents.AgentState` class, which hold our custom fields - this is similar to a dataclass we have see so far

```python
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_color: str
```

2. We can update the state inside of our tool function as below:

```python
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_color(color: str, runtime: ToolRuntime) -> Command:
    """update the favourite color of user once they have revealed it"""
    return Command(update={
        "favourite_color" : color,
        "messages": [
            ToolMessage(
                f"Successfully updated favourite color from {runtime.context.favourite_color} to {color}",tool_call_id=runtime.tool_call_id
            ),
        ],
    })
```

In [65]:
from langchain.agents import AgentState


# NOTE: you CANNOT set default values for fields of AgentState!!!
class CustomState(AgentState):
    favourite_color: str
    least_favourite_color: str

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """gets favourite color of user"""
    print(f" -------- get_favourite_color({runtime.state}) tool called --------")
    return runtime.state["favourite_color"]


@tool
def get_least_favourite_color(runtime: ToolRuntime) -> str:
    """gets favourite color of user"""
    print(f" -------- get_least_favourite_color({runtime.state}) tool called --------")
    return runtime.state["least_favourite_color"]


@tool
def update_favourite_color(color: str, runtime: ToolRuntime) -> Command:
    """update the favourite color of user once they have revealed it"""
    return Command(
        update={
            "favourite_color": color,
            "messages": [
                ToolMessage(
                    f"Successfully updated favourite color from {runtime.context.favourite_color} to {color}",
                    tool_call_id=runtime.tool_call_id,
                ),
            ],
        }
    )

In [67]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# color_pref = ColorContext(favourite_color="blue", least_favourite_color="yellow")
state_schema = CustomState(favourite_color="blue", least_favourite_color="yellow")

agent2 = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=(
        """
        You are a helpful agent that can answer information about my color preference.
        You have access to the following tools:
            - get_favourite_color : to get my favourite color
            - get_least_favourite_color : to get my least favourite color
            - update_favourite_color : to update favourite color should user ask you to
        Use these tools for any question regarding my color preference & return response as
            "Your [least]favourite color is <<color>>"
        For all other queries about me, respond as follows:
            "Sorry, this information is confidential. I can only tell you about your color preference"
        """
    ),
    tools=[get_favourite_color, get_least_favourite_color, update_favourite_color],
    # here you pass in updateable schema
    state_schema=CustomState,
)

In [69]:
config2 = {"configurable": {"thread_id": "12634567890"}}

response = agent2.invoke(
    {"messages": {"role": "user", "content": "Set my favourite color to aqua"}},
    config=config2,
    context=color_pref,
)
# I am printing all response messages here!
console.print(response)

c:\Dev\Code\git-projects\learning_langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


{
    'messages': [
        HumanMessage(
            content='Set my favourite color to aqua',
            additional_kwargs={},
            response_metadata={},
            id='8090543c-e358-4cc8-84fd-368f5d29dbfe'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 922,
                    'prompt_tokens': 307,
                    'total_tokens': 1229,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 896,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-DW28WadlWIhbBIoHhBnDQPpqWSoXM',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019da143-f8f4-7b60-8c05-cec57a982b5d-0',
            tool_calls=[
                {
                    'name': 'update_favourite_color',
                    'args': {'color': 'aqua'},
                    'id': 'call_SdfIVFU1l1tMbDNVZ9F5qE1n',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 307,
                'output_tokens': 922,
                'total_tokens': 1229,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 896}
            }
        ),
        ToolMessage(
            content='Successfully updated favourite color from blue to aqua',
            name='update_favourite_color',
            id='12b32eb4-ddfc-4120-8978-ec9aafc70f90',
            tool_call_id='call_SdfIVFU1l1tMbDNVZ9F5qE1n'
        ),
        AIMessage(
            content='Your [least]favourite color is <<color>>',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 14,
                    'prompt_tokens': 348,
                    'total_tokens': 362,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-DW28dlN5ZMOatku45hiE8kZjF1MLF',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019da144-14a7-75b3-808d-a22f1481c43c-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 14,
                'total_tokens': 362,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'favourite_color': 'aqua'
}

In [70]:
# since agent has memory, we need to pass in a config to invoke()
response = agent2.invoke(
    {"messages": {"role": "user", "content": "What is my favourite color?"}},
    config=config2,
    context=color_pref,
)
console.print(f"[bright_blue]{response["messages"][-1].content}[/bright_blue]")

 -------- get_favourite_color(ColorContext(favourite_color='blue', least_favourite_color='yellow')) tool called --------


c:\Dev\Code\git-projects\learning_langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


KeyError: 'favourite_color'

In [59]:
response = agent2.invoke(
    {"messages": {"role": "user", "content": "Set my favourite color to aqua"}},
    config=config2,
    context=color_pref,
)
# I am printing all response messages here!
console.print(response)

c:\Dev\Code\git-projects\learning_langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


{
    'messages': [
        HumanMessage(
            content='Set my favourite color to aqua',
            additional_kwargs={},
            response_metadata={},
            id='5ce1c33e-4456-4beb-96cc-6abbac846316'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 666,
                    'prompt_tokens': 299,
                    'total_tokens': 965,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 640,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-DW20CaqYEd9WfaQ2zM6b4hIDypL4B',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019da13c-1bf0-7070-938c-77957eb767fe-0',
            tool_calls=[
                {
                    'name': 'update_favourite_color',
                    'args': {'color': 'aqua'},
                    'id': 'call_tsNaOzVXcl88ydP4pHe4Ej3O',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 299,
                'output_tokens': 666,
                'total_tokens': 965,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 640}
            }
        ),
        ToolMessage(
            content='Successfully updated favourite color from blue to aqua',
            name='update_favourite_color',
            id='8e20f3af-703b-4726-8543-41fef75ba8e5',
            tool_call_id='call_tsNaOzVXcl88ydP4pHe4Ej3O'
        ),
        AIMessage(
            content='Your favourite color is aqua',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 462,
                    'prompt_tokens': 340,
                    'total_tokens': 802,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 448,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-DW20IEArz96U2ylGd5SmoiA7TDqZi',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019da13c-340c-7c40-afc7-bf584886b6a5-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 340,
                'output_tokens': 462,
                'total_tokens': 802,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 448}
            }
        )
    ],
    'favourite_color': 'aqua'
}

In [60]:
# now let's ask again for favourite color
response = agent.invoke(
    {"messages": {"role": "user", "content": "What is my favourite color?"}},
    config=config2,
    context=color_pref,
)
console.print(f"[bright_blue]{response["messages"][-1].content}[/bright_blue]")

c:\Dev\Code\git-projects\learning_langchain\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


 -------- favourite_color(ColorContext(favourite_color='blue', least_favourite_color='yellow')) tool called --------


Your favourite color is blue